# 03. Sequence to Sequence Learning with Neural Networks (Sutskever et al., 2014)

Interactive walkthrough accompanying `03_seq2seq_encoder_decoder.py`.

We cover:
1. Building a 2-layer stacked LSTM from raw tensor operations (`x @ W_ih + h @ W_hh + b`).
2. How the encoder compresses the entire input into a fixed-size state bottleneck (`c_enc`).
3. Teacher forcing during training vs. greedy autoregressive decoding during inference.
4. Shape, step-equivalence, and gradient-flow verification.
5. Demonstrating the Sutskever et al. (2014) source-reversal trick and fixed-context bottleneck degradation from $T = 8$ to $T = 20$.

## 1. Setup

Import PyTorch and Matplotlib, and fix the random seed for reproducibility.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.set_num_threads(1)
torch.manual_seed(42)

PAD_IDX, SOS_IDX, EOS_IDX = 0, 1, 2
print("PyTorch version:", torch.__version__)

## 2. Encoder-Decoder Equations & Intuition

A Seq2Seq model factorizes the conditional probability of an output sequence $y_1, \dots, y_T$ given an input sequence $x_1, \dots, x_S$ using two separate multi-layer LSTMs:

### 2.1 Encoder & Fixed-Size Context Bottleneck ($c_{\text{enc}}$)

At each source step $t = 1, \dots, S$, the $L=2$ stacked encoder updates layer states $(\mathbf{h}_t^{(\ell)}, \mathbf{c}_t^{(\ell)})$:

$$(\mathbf{h}_t^{(1)}, \mathbf{c}_t^{(1)}) = \text{LSTMCell}_{\text{enc}}^{(1)}\big(\mathbf{e}(x_t),\, (\mathbf{h}_{t-1}^{(1)}, \mathbf{c}_{t-1}^{(1)})\big)$$
$$(\mathbf{h}_t^{(2)}, \mathbf{c}_t^{(2)}) = \text{LSTMCell}_{\text{enc}}^{(2)}\big(\mathbf{h}_t^{(1)},\, (\mathbf{h}_{t-1}^{(2)}, \mathbf{c}_{t-1}^{(2)})\big)$$

After the final source token $x_S$, every intermediate hidden vector is discarded. Only the final state tuple

$$\mathbf{c}_{\text{enc}} = \Big\{(\mathbf{h}_S^{(1)}, \mathbf{c}_S^{(1)}),\; (\mathbf{h}_S^{(2)}, \mathbf{c}_S^{(2)})\Big\}$$

is passed to initialize the decoder. Because $\mathbf{c}_{\text{enc}}$ has fixed dimension regardless of $S$, it forms an **information bottleneck** as sequences grow longer.

### 2.2 Teacher Forcing (Training)

During training, the decoder is initialized with $\mathbf{c}_{\text{enc}}$ and fed the shifted ground-truth prefix $(\langle\text{SOS}\rangle, y_1, \dots, y_{T-1})$:

$$P(y_t \mid y_{<t}, \mathbf{x}) = \text{softmax}\big(\mathbf{W}_{\text{out}}\, \mathbf{h}_t^{\text{dec}, (2)} + \mathbf{b}_{\text{out}}\big)$$

* **Intuition:** Feeding ground-truth $y_{t-1}$ keeps every step aligned with the true target prefix so early mistakes do not derail gradients for later tokens.

### 2.3 Greedy Autoregressive Decoding (Inference)

At inference time, true targets are unknown. Starting from $\hat{y}_0 = \langle\text{SOS}\rangle$, the decoder feeds its own argmax token back into the next step until $\langle\text{EOS}\rangle$ or `max_len`:

$$\hat{y}_t = \arg\max_{v}\; P(y_t = v \mid \hat{y}_{<t}, \mathbf{x})$$

* **Intuition:** Errors compound autoregressively (exposure bias): if $\hat{y}_1$ is wrong, all subsequent states drift away from the distribution seen during teacher-forced training.

In [ ]:
class LSTMCellFromScratch(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, forget_bias_init: float = 1.0):
        super().__init__()
        self.input_dim = input_dim
        self.hidden_dim = hidden_dim
        self.W_ih = nn.Parameter(torch.empty(input_dim, 4 * hidden_dim))
        self.W_hh = nn.Parameter(torch.empty(hidden_dim, 4 * hidden_dim))
        self.b = nn.Parameter(torch.zeros(4 * hidden_dim))
        bound = 1.0 / (hidden_dim ** 0.5)
        nn.init.uniform_(self.W_ih, -bound, bound)
        nn.init.uniform_(self.W_hh, -bound, bound)
        nn.init.uniform_(self.b, -bound, bound)
        with torch.no_grad():
            self.b[hidden_dim : 2 * hidden_dim].fill_(forget_bias_init)

    def forward(self, x_t: torch.Tensor, state=None):
        B = x_t.shape[0]
        if state is None:
            h_prev = torch.zeros(B, self.hidden_dim, device=x_t.device, dtype=x_t.dtype)
            c_prev = torch.zeros(B, self.hidden_dim, device=x_t.device, dtype=x_t.dtype)
        else:
            h_prev, c_prev = state

        gates = x_t @ self.W_ih + h_prev @ self.W_hh + self.b
        i_raw, f_raw, g_raw, o_raw = gates.chunk(4, dim=-1)
        c_t = torch.sigmoid(f_raw) * c_prev + torch.sigmoid(i_raw) * torch.tanh(g_raw)
        h_t = torch.sigmoid(o_raw) * torch.tanh(c_t)
        return h_t, c_t


class StackedLSTM(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, num_layers: int = 2, forget_bias_init: float = 1.0):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        self.layers = nn.ModuleList([
            LSTMCellFromScratch(input_dim if l == 0 else hidden_dim, hidden_dim, forget_bias_init)
            for l in range(num_layers)
        ])

    def init_states(self, batch_size: int, device, dtype):
        return [
            (torch.zeros(batch_size, self.hidden_dim, device=device, dtype=dtype),
             torch.zeros(batch_size, self.hidden_dim, device=device, dtype=dtype))
            for _ in range(self.num_layers)
        ]

    def step(self, x_t: torch.Tensor, states=None):
        if states is None:
            states = self.init_states(x_t.shape[0], x_t.device, x_t.dtype)
        h_in = x_t
        next_states = []
        for l, cell in enumerate(self.layers):
            h_l, c_l = cell(h_in, states[l])
            next_states.append((h_l, c_l))
            h_in = h_l
        return h_in, next_states

    def forward(self, x: torch.Tensor, states=None):
        B, T, _ = x.shape
        if states is None:
            states = self.init_states(B, x.device, x.dtype)
        outputs = []
        for t in range(T):
            top_h, states = self.step(x[:, t, :], states)
            outputs.append(top_h)
        return torch.stack(outputs, dim=1), states


class Encoder(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, num_layers: int = 2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm = StackedLSTM(embed_dim, hidden_dim, num_layers=num_layers)

    def forward(self, src_tokens: torch.Tensor):
        _, final_states = self.lstm(self.embedding(src_tokens))
        return final_states


class Decoder(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, num_layers: int = 2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.lstm = StackedLSTM(embed_dim, hidden_dim, num_layers=num_layers)
        self.proj = nn.Linear(hidden_dim, vocab_size)

    def step(self, token_t: torch.Tensor, states):
        top_h, next_states = self.lstm.step(self.embedding(token_t), states)
        return self.proj(top_h), next_states

    def forward_teacher_forcing(self, tgt_in: torch.Tensor, encoder_states):
        states = encoder_states
        logits_seq = []
        for t in range(tgt_in.shape[1]):
            logits_t, states = self.step(tgt_in[:, t], states)
            logits_seq.append(logits_t)
        return torch.stack(logits_seq, dim=1)

    @torch.no_grad()
    def greedy_decode(self, encoder_states, max_len: int):
        B = encoder_states[0][0].shape[0]
        device = encoder_states[0][0].device
        states = encoder_states
        cur = torch.full((B,), SOS_IDX, dtype=torch.long, device=device)
        finished = torch.zeros(B, dtype=torch.bool, device=device)
        preds, logits_seq = [], []
        for _ in range(max_len):
            logits_t, states = self.step(cur, states)
            next_tok = logits_t.argmax(dim=-1)
            preds.append(torch.where(finished, torch.full_like(next_tok, PAD_IDX), next_tok))
            logits_seq.append(logits_t)
            finished = finished | (next_tok == EOS_IDX)
            cur = next_tok
            if finished.all():
                break
        return torch.stack(preds, dim=1), torch.stack(logits_seq, dim=1)


class Seq2Seq(nn.Module):
    def __init__(self, vocab_size: int, embed_dim: int, hidden_dim: int, num_layers: int = 2):
        super().__init__()
        self.encoder = Encoder(vocab_size, embed_dim, hidden_dim, num_layers=num_layers)
        self.decoder = Decoder(vocab_size, embed_dim, hidden_dim, num_layers=num_layers)

    def forward(self, src_tokens: torch.Tensor, tgt_in: torch.Tensor):
        return self.decoder.forward_teacher_forcing(tgt_in, self.encoder(src_tokens))

    @torch.no_grad()
    def greedy_decode(self, src_tokens: torch.Tensor, max_len: int):
        return self.decoder.greedy_decode(self.encoder(src_tokens), max_len=max_len)

print("2-layer Seq2Seq modules ready.")

## 3. Shape & Teacher-Forcing vs. Greedy Decoding Sanity Check

Verify three invariants before training:
1. `Encoder` returns 2 layer tuples `(h_l, c_l)` each shaped `(B, H)`.
2. `forward_teacher_forcing` matches manual single-step `decoder.step(...)` calls to `< 1e-6`, and matches step `t = 0` of `greedy_decode` (`<SOS>` input).
3. Gradients flow cleanly through both decoder and encoder layers.

In [ ]:
def make_translation_batch(batch_size: int, seq_len: int, vocab_size: int = 11, shift: int = 3, reverse_src: bool = False):
    num_content = vocab_size - 3
    x = torch.randint(3, vocab_size, (batch_size, seq_len))
    y = 3 + ((x - 3 + shift) % num_content)
    src = torch.flip(x, dims=[1]) if reverse_src else x
    sos = torch.full((batch_size, 1), SOS_IDX, dtype=torch.long)
    eos = torch.full((batch_size, 1), EOS_IDX, dtype=torch.long)
    return src, torch.cat([sos, y], dim=1), torch.cat([y, eos], dim=1)

B, T, V, E, H, L = 4, 6, 11, 16, 32, 2
check_model = Seq2Seq(V, E, H, num_layers=L)
src_c, tgt_in_c, tgt_out_c = make_translation_batch(B, T, vocab_size=V, reverse_src=True)

enc_states = check_model.encoder(src_c)
assert len(enc_states) == L and all(h.shape == (B, H) and c.shape == (B, H) for h, c in enc_states)

tf_logits = check_model(src_c, tgt_in_c)
assert tf_logits.shape == (B, T + 1, V)

s = enc_states
manual_steps = []
for t in range(tgt_in_c.shape[1]):
    lt, s = check_model.decoder.step(tgt_in_c[:, t], s)
    manual_steps.append(lt)
tf_diff = (tf_logits - torch.stack(manual_steps, dim=1)).abs().max().item()

check_model.eval()
preds_c, greedy_logits_c = check_model.greedy_decode(src_c, max_len=T + 1)
t0_diff = (greedy_logits_c[:, 0, :] - tf_logits[:, 0, :]).abs().max().item()

loss_c = F.cross_entropy(tf_logits.reshape(-1, V), tgt_out_c.reshape(-1))
loss_c.backward()
enc_g = check_model.encoder.lstm.layers[0].W_hh.grad.norm().item()
dec_g = check_model.decoder.lstm.layers[1].W_hh.grad.norm().item()

print(f"tf_step_diff={tf_diff:.2e} | greedy_t0_diff={t0_diff:.2e} | enc_grad={enc_g:.3e} | dec_grad={dec_g:.3e}")
assert tf_diff < 1e-6 and t0_diff < 1e-6 and enc_g > 0.0 and dec_g > 0.0

## 4. Sutskever et al. (2014) Source-Reversal Experiment ($T = 16$)

On an order-preserving token-translation task (`y_t = (x_t + 3) mod 8`), feeding the source forward `[x_1, ..., x_T]` places $T$ steps between $x_1$ and $y_1$. Reversing the source to `[x_T, ..., x_1]` puts $x_1$ directly adjacent to $y_1$ across the bottleneck.

Train both orderings at $T = 16$ ($H = 64$, 2 layers) and plot both the **training loss curves** and **per-position greedy accuracy**.

In [ ]:
def train_and_eval(seq_len: int, reverse_src: bool, vocab_size: int = 11, hidden_dim: int = 64, steps: int = 450, lr: float = 5e-3):
    torch.manual_seed(42)
    model = Seq2Seq(vocab_size=vocab_size, embed_dim=24, hidden_dim=hidden_dim, num_layers=2)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for _ in range(steps):
        model.train()
        src, tgt_in, tgt_out = make_translation_batch(64, seq_len, vocab_size=vocab_size, reverse_src=reverse_src)
        logits = model(src, tgt_in)
        loss = F.cross_entropy(logits.reshape(-1, vocab_size), tgt_out.reshape(-1))
        opt.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        src_v, tgt_in_v, tgt_out_v = make_translation_batch(256, seq_len, vocab_size=vocab_size, reverse_src=reverse_src)
        tf_loss = F.cross_entropy(model(src_v, tgt_in_v).reshape(-1, vocab_size), tgt_out_v.reshape(-1)).item()
        max_len = tgt_out_v.shape[1]
        preds, _ = model.greedy_decode(src_v, max_len=max_len)
        if preds.shape[1] < max_len:
            pad = torch.full((256, max_len - preds.shape[1]), PAD_IDX, dtype=torch.long)
            preds = torch.cat([preds, pad], dim=1)
        pos_acc = (preds[:, :seq_len] == tgt_out_v[:, :seq_len]).float().mean(dim=0).numpy()
        tok_acc = float(pos_acc.mean())
        seq_acc = (preds == tgt_out_v).all(dim=1).float().mean().item()
    return model, losses, pos_acc, (tf_loss, tok_acc, seq_acc)

m_norm_16, loss_norm_16, pos_norm_16, met_norm_16 = train_and_eval(16, reverse_src=False)
m_rev_16, loss_rev_16, pos_rev_16, met_rev_16 = train_and_eval(16, reverse_src=True)

print(f"T=16 Normal   | tf_loss={met_norm_16[0]:.4f} | tok_acc={met_norm_16[1]:.3f} | seq_acc={met_norm_16[2]:.3f}")
print(f"T=16 Reversed | tf_loss={met_rev_16[0]:.4f} | tok_acc={met_rev_16[1]:.3f} | seq_acc={met_rev_16[2]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(loss_norm_16, label="Normal source", color="#e74c3c", alpha=0.85)
axes[0].plot(loss_rev_16, label="Reversed source", color="#2980b9", alpha=0.9)
axes[0].set_title("Training Cross-Entropy Loss (T = 16)")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("Loss")
axes[0].grid(True, alpha=0.4)
axes[0].legend()

positions = list(range(1, 17))
axes[1].plot(positions, pos_norm_16, marker="o", label="Normal source", color="#e74c3c", lw=2)
axes[1].plot(positions, pos_rev_16, marker="s", label="Reversed source", color="#2980b9", lw=2)
axes[1].set_title("Greedy Accuracy by Output Position (T = 16)")
axes[1].set_xlabel("Target position t")
axes[1].set_ylabel("Token Accuracy")
axes[1].set_ylim(-0.05, 1.05)
axes[1].grid(True, alpha=0.4)
axes[1].legend()
plt.tight_layout()
plt.show()

## 5. Fixed-Context Bottleneck Degradation ($T = 8$ vs. $T = 20$)

Compare greedy decoding accuracy at $T = 8$, $T = 16$, and $T = 20$ with the same fixed $H = 64$ bottleneck. Notice how accuracy collapses as sequence length grows, motivating the attention mechanism explored next.

In [ ]:
lengths = [8, 16, 20]
norm_tok, rev_tok = [], []
norm_seq, rev_seq = [], []
models_rev = {}

for T_len in lengths:
    if T_len == 16:
        mn, mr = met_norm_16, met_rev_16
        models_rev[T_len] = m_rev_16
    else:
        _, _, _, mn = train_and_eval(T_len, reverse_src=False)
        mr_model, _, _, mr = train_and_eval(T_len, reverse_src=True)
        models_rev[T_len] = mr_model
    norm_tok.append(mn[1])
    rev_tok.append(mr[1])
    norm_seq.append(mn[2])
    rev_seq.append(mr[2])
    print(f"T={T_len:2d} | Normal tok={mn[1]:.3f} seq={mn[2]:.3f} | Reversed tok={mr[1]:.3f} seq={mr[2]:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(lengths, norm_tok, marker="o", label="Normal source", color="#e74c3c", lw=2)
axes[0].plot(lengths, rev_tok, marker="s", label="Reversed source", color="#2980b9", lw=2)
axes[0].set_title("Token Accuracy vs. Sequence Length (H = 64)")
axes[0].set_xlabel("Sequence length T")
axes[0].set_ylabel("Greedy Token Accuracy")
axes[0].set_xticks(lengths)
axes[0].set_ylim(-0.05, 1.05)
axes[0].grid(True, alpha=0.4)
axes[0].legend()

axes[1].plot(lengths, norm_seq, marker="o", label="Normal source", color="#e74c3c", lw=2)
axes[1].plot(lengths, rev_seq, marker="s", label="Reversed source", color="#2980b9", lw=2)
axes[1].set_title("Exact Sequence Match vs. Sequence Length")
axes[1].set_xlabel("Sequence length T")
axes[1].set_ylabel("Greedy Sequence Accuracy")
axes[1].set_xticks(lengths)
axes[1].set_ylim(-0.05, 1.05)
axes[1].grid(True, alpha=0.4)
axes[1].legend()
plt.tight_layout()
plt.show()

# Inspect one decoded example at T=8 vs T=20 (reversed source).
for T_ex in [8, 20]:
    s_ex, _, tgt_ex = make_translation_batch(1, T_ex, reverse_src=True)
    p_ex, _ = models_rev[T_ex].greedy_decode(s_ex, max_len=T_ex + 1)
    print(f"T={T_ex:2d} target : {tgt_ex[0].tolist()}")
    print(f"T={T_ex:2d} decoded: {p_ex[0].tolist()}")